# VigiLens on Google Colab (T4)

This notebook trains and evaluates VigiLens end to end:

1. Check the GPU and enable FP16 AMP.
2. Clone the repository and install dependencies.
3. Create a synthetic smoke dataset **or** prepare OTCBVS / KAIST.
4. Precompute RGB/IR features once.
5. Train the fusion + causal LSTM on cached features.
6. Calibrate the anomaly threshold on normal validation data.
7. Evaluate frame-level AUC/AP/FPR and export the results.

**Runtime:** select `Runtime -> Change runtime type -> T4 GPU` before running.

> The synthetic path is a plumbing test, not a benchmark. Use OTCBVS as a first real-data preview and KAIST as a larger normal-pedestrian corpus. Anomaly metrics need a dataset with frame-level anomaly labels (ShanghaiTech/Avenue or custom labeled clips).

In [ ]:
import os, sys, json, shutil
from pathlib import Path

import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("PyTorch:", torch.__version__)
print("Device:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    print("Capability:", torch.cuda.get_device_capability(0))
    torch.backends.cudnn.benchmark = True
else:
    print("No GPU detected. Cached training still works on CPU, but it will be slower.")

In [ ]:
# Change this if you forked the repository.
REPO_URL = "https://github.com/soumyaranjan4446/VigiLens.git"
PROJECT_DIR = "/content/VigiLens"

if not Path(PROJECT_DIR).exists():
    !git clone {REPO_URL} {PROJECT_DIR}

%cd {PROJECT_DIR}
!git rev-parse --short HEAD

In [ ]:
# Colab already provides a CUDA build of torch/torchvision; pip keeps it.
!pip install -q -r requirements.txt

import sklearn, cv2, ultralytics, streamlit
print("opencv", cv2.__version__)
print("ultralytics", ultralytics.__version__)
print("streamlit", streamlit.__version__)

In [ ]:
# Optional: persist checkpoints and results in Google Drive.
USE_DRIVE = False
DRIVE_DIR = None

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_DIR = Path("/content/drive/MyDrive/VigiLens_runs")
    DRIVE_DIR.mkdir(parents=True, exist_ok=True)
    print("Drive export directory:", DRIVE_DIR)
else:
    print("Drive export disabled. Set USE_DRIVE=True to enable it.")

## Option A: Synthetic smoke test (fast)

This creates short RGB/IR clips under `Data/` and a labeled evaluation manifest under `Evaluation/`.

In [ ]:
!python scripts/make_synthetic_dataset.py \
    --output Data \
    --eval-dir Evaluation \
    --frames 96 \
    --num-train 4 \
    --num-val 2 \
    --num-normal-eval 2 \
    --num-anomaly 2 \
    --overwrite

## Option B: OTCBVS Dataset 10 (first real-data preview)

Upload or download OTCBVS into `/content/OTCBVS`. The preparation script discovers visible/thermal video pairs and paired image directories. For a quick preview, keep `--max-frames 300`.

```bash
!python scripts/prepare_otcbvs.py --input /content/OTCBVS --output Data --max-frames 300
```

## Option C: KAIST Multispectral Pedestrian (main normal corpus)

KAIST must be organized as `set00/V000/visible/*.jpg` and `set00/V000/lwir/*.jpg`.

```bash
!python scripts/prepare_kaist.py --input /content/KAIST --output Data --sets 00 --val-ratio 0.2 --max-frames 300
```

KAIST has no frame-level anomaly labels; use it for normal-only training and evaluate on labeled clips.

In [ ]:
# Run this only if you prepared OTCBVS or KAIST above.
# It is safe to leave commented out for the synthetic path.

# !python scripts/prepare_otcbvs.py --input /content/OTCBVS --output Data --max-frames 300
# !python scripts/prepare_kaist.py --input /content/KAIST --output Data --sets 00 --val-ratio 0.2 --max-frames 300

## Real Data: OTCBVS Dataset 10 (automatic download)

This section runs the full real-data pipeline: download OTCBVS, prepare videos, precompute features (with YOLO), train longer, calibrate, and evaluate.

In [ ]:
!wget -q -O BilodeauetAlInfraredDataset.zip http://www.polymtl.ca/litiv/vid/BilodeauetAlInfraredDataset.zip
!unzip -o BilodeauetAlInfraredDataset.zip -d /content/OTCBVS

In [ ]:
!rm -rf Data cache
!python scripts/prepare_otcbvs.py --input /content/OTCBVS --output Data --max-frames 300

In [ ]:
!python -m training.precompute --data-dir Data --output cache --device {DEVICE} --batch-size 16 --overwrite

In [ ]:
!python -m training.train --cache-dir cache --epochs 30 --batch-size 8 --seq-len 16 --device {DEVICE} --checkpoint-dir checkpoints

In [ ]:
!python -m training.calibration --cache-dir cache/val --checkpoint checkpoints/best_model.pth --output checkpoints/calibration.json --device {DEVICE} --k-std 3.0 --percentile 95

In [ ]:
!python -m training.evaluate --manifest Evaluation/manifest.json --cache-dir cache/eval --checkpoint checkpoints/best_model.pth --calibration checkpoints/calibration.json --output-dir Evaluation/results --device {DEVICE} --smoothing-window 5

In [ ]:
import shutil
from pathlib import Path
for name in ["checkpoints", "Evaluation/results"]:
    src = Path(name)
    dst = Path("/content/drive/MyDrive/VigiLens_runs") / name
    if src.exists():
        dst.parent.mkdir(parents=True, exist_ok=True)
        if dst.exists():
            shutil.rmtree(dst)
        shutil.copytree(src, dst)
        print(f"Copied {src} -> {dst}")

## Save full model for the Streamlit app

The trained checkpoint (`checkpoints/best_model.pth`) contains the fusion + temporal weights, and (`checkpoints/calibration.json`) the calibrated threshold. The Streamlit app loads both and holds the frozen ResNet encoders, so this trained model is directly usable with the app.

## Precompute features (synthetic smoke-test fallback)

The first run downloads `yolov8n-seg.pt` and ImageNet ResNet50 weights. `--max-frames 300` keeps the preview quick; remove it for a full run. Use `--no-segmentation` only for the synthetic smoke test or when YOLO misses the subjects.

In [ ]:
# Synthetic smoke test: full-frame features are enough.
!python -m training.precompute \
    --data-dir Data \
    --output cache \
    --device {DEVICE} \
    --batch-size 16 \
    --max-frames 300 \
    --no-segmentation

In [ ]:
# For OTCBVS/KAIST, use the real YOLO path instead:
# !python -m training.precompute --data-dir Data --output cache --device {DEVICE} --batch-size 16 --max-frames 300

In [ ]:
# Precompute the labeled evaluation manifest (synthetic or custom).
!python -m training.precompute \
    --manifest Evaluation/manifest.json \
    --output cache/eval \
    --device {DEVICE} \
    --batch-size 16 \
    --max-frames 300 \
    --no-segmentation

## Train fusion + LSTM on cached features (synthetic fallback)

The ResNet encoders stay frozen (and are not even instantiated in this path). AMP is enabled automatically on CUDA.

In [ ]:
!python -m training.train \
    --cache-dir cache \
    --epochs 5 \
    --batch-size 8 \
    --seq-len 16 \
    --device {DEVICE} \
    --checkpoint-dir checkpoints

## Calibrate the threshold on normal validation caches (synthetic fallback)

In [ ]:
!python -m training.calibration \
    --cache-dir cache/val \
    --checkpoint checkpoints/best_model.pth \
    --output checkpoints/calibration.json \
    --device {DEVICE} \
    --k-std 3.0 \
    --percentile 95

## Evaluate with frame-level metrics (synthetic fallback)

In [ ]:
!python -m training.evaluate \
    --manifest Evaluation/manifest.json \
    --cache-dir cache/eval \
    --checkpoint checkpoints/best_model.pth \
    --calibration checkpoints/calibration.json \
    --output-dir Evaluation/results \
    --device {DEVICE} \
    --smoothing-window 5

In [ ]:
import pandas as pd

results = json.loads(Path("Evaluation/results/results.json").read_text(encoding="utf-8"))
summary = {
    "videos": results["num_videos"],
    "frames": results["num_frames"],
    "threshold": results["threshold"],
    "auc": results["metrics"]["auc"],
    "ap": results["metrics"]["ap"],
    "precision": results["metrics"]["precision"],
    "recall": results["metrics"]["recall"],
    "f1": results["metrics"]["f1"],
    "fpr": results["metrics"]["fpr"],
    "fpr_at_95tpr": results["metrics"]["fpr_at_95tpr"],
}
print(json.dumps(summary, indent=2))

scores = pd.read_csv("Evaluation/results/scores.csv")
display(scores.head())
display(scores.groupby("sequence_id")["smoothed_score"].agg(["count", "mean", "max"]))

## Run the Streamlit app (optional)

Colab cannot expose port 8501 directly. Use a tunnel such as `cloudflared` or `ngrok` if you want the UI:

```bash
!pip install -q streamlit cloudflared
!streamlit run app.py --server.port 8501 --server.address 0.0.0.0 &
# then start a tunnel to localhost:8501 in another cell
```

The app needs `checkpoints/best_model.pth`, `checkpoints/calibration.json`, and `yolov8n-seg.pt` (downloaded automatically on first use).

## Next steps

- Replace the synthetic evaluation set with labeled anomaly clips.
- Recalibrate whenever the camera/domain changes.
- Train longer on KAIST normal data, then fine-tune stage 4 with `--online --unfreeze-layer4` if GPU memory allows.
- Compare `metrics_raw` and `metrics_smoothed`; smoothing trades latency for stability.